# Residual-Based Feature Boosting PoC

VS Code/Jupyter에서 위에서 아래로 실행하는 노트북입니다.

핵심 흐름:

1. `Xb` base feature로 baseline 수율 회귀 모델 학습
2. `baseline_residual = y - baseline_pred` 계산
3. defect별 bad/good group 기준으로 candidate feature 품질 필터링
4. candidate feature 하나만 사용해 current residual 예측
5. validation bad group의 `bad_rmse_reduction` 기준으로 feature 선택
6. 선택 feature를 `Xb + selected Xnew`에 추가해 final CatBoost 재학습
7. baseline vs final metric, ranking, SHAP summary 저장

중요: residual boosting 단계에서는 `Xb`를 다시 학습하지 않습니다. 후보 feature `x_j` 하나만 residual model에 사용합니다.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "feature_boosting").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("repo root not found")
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from feature_boosting.baseline_model import add_baseline_predictions, metrics_by_split, train_baseline_model
from feature_boosting.data_loader import align_base_and_candidates, candidate_feature_cols, load_base_dataset, load_base_feature_cols, load_candidate_features, load_group_ids
from feature_boosting.final_model import evaluate_model_by_groups, predict_final, train_final_model
from feature_boosting.reporting import baseline_residual_summary, plot_residual_curve, prepare_output_dir, write_csv
from feature_boosting.residual_boosting import ResidualFeatureBooster, ResidualFeatureBoosterConfig
from feature_boosting.shap_analysis import compute_shap_summary
from feature_boosting.splitter import split_frame
from feature_boosting.validation import profile_candidate_features, validate_defect_groups, validate_input_columns
from feature_boosting.config import FeatureFilterConfig

print("ROOT =", ROOT)

## 1. 실험 설정

기본값은 `USE_DEMO_DATA = True`입니다. 그래서 repo를 clone한 직후 실제 데이터가 없어도 toyset을 자동 생성해서 바로 실행됩니다.

실제 데이터가 준비되어 있으면 `USE_DEMO_DATA = False`로 바꾸고 아래 경로만 수정하면 됩니다.

CatBoost가 설치되어 있으면 `backend: "auto"`에서 CatBoost를 사용합니다. CatBoost만 강제하려면 `backend: "catboost"`로 바꾸세요.

In [ ]:
USE_DEMO_DATA = True
DEMO_N_WAFERS = 14_000
DEMO_N_CANDIDATE_FEATURES = 5_000  # hidden_defect_1/2 포함 총 candidate feature 수
DEMO_RANDOM_SEED = 42
RUN_ID = "feature_boosting_rev0_notebook"

ID_COL = "sample_id"
TARGET_COL = "yield"
SPLIT_COL = "split"

BASE_DATASET_PATH = ROOT / "data" / "base_dataset.parquet"
CANDIDATE_FEATURES_PATH = ROOT / "data" / "candidate_features.parquet"
BASE_FEATURE_COLS_PATH = ROOT / "data" / "base_feature_cols.txt"
OUTPUT_BASE_DIR = ROOT / "outputs"

DEFECTS = [
    {
        "defect_id": "defect_1",
        "bad_group_path": ROOT / "data" / "groups" / "defect_1_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_1_good.csv",
    },
    {
        "defect_id": "defect_2",
        "bad_group_path": ROOT / "data" / "groups" / "defect_2_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_2_good.csv",
    },
    {
        "defect_id": "defect_3",
        "bad_group_path": ROOT / "data" / "groups" / "defect_3_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_3_good.csv",
    },
]

BASELINE_MODEL_PARAMS = {
    "backend": "auto",
    "iterations": 3000,
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "early_stopping_rounds": 100,
    "verbose": 200,
}

RESIDUAL_MODEL_PARAMS = {
    "backend": "auto",
    "iterations": 300,
    "depth": 3,
    "learning_rate": 0.05,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "early_stopping_rounds": 30,
    "verbose": False,
    "thread_count": 1,
}

FINAL_MODEL_PARAMS = dict(BASELINE_MODEL_PARAMS)

FEATURE_FILTER = FeatureFilterConfig(
    max_missing_rate=0.5,
    min_unique_values=2,
    min_bad_coverage=0.7,
    min_good_coverage=0.7,
)

BOOSTING_N_ROUNDS = 5
SELECT_PER_ROUND = 1
MIN_IMPROVEMENT = 0.0
MIN_VALID_BAD_SAMPLES = 1
SHAP_ENABLED = True
SHAP_MAX_SAMPLES = 5000

OUT_DIR = prepare_output_dir(OUTPUT_BASE_DIR, RUN_ID)
print("OUT_DIR =", OUT_DIR)

## 2. 선택 사항: 데모 데이터 생성

`USE_DEMO_DATA = True`이면 아래 셀이 toyset CSV를 자동으로 생성합니다. clone 직후에는 이 기본값 그대로 실행하면 됩니다.

In [ ]:
if USE_DEMO_DATA:
    demo_dir = ROOT / "data" / "residual_poc_demo"
    group_dir = demo_dir / "groups"
    group_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(DEMO_RANDOM_SEED)
    n = DEMO_N_WAFERS
    n_noise_features = max(0, DEMO_N_CANDIDATE_FEATURES - 2)
    sample_id = np.array([f"WF_{idx:04d}" for idx in range(n)])
    split = np.array(["train"] * 360 + ["valid"] * 120 + ["test"] * 120)
    rng.shuffle(split)

    base_temp = rng.normal(0, 1, n)
    base_pressure = rng.normal(0, 1, n)
    hidden_defect_1 = rng.normal(0, 1, n)
    hidden_defect_2 = rng.normal(0, 1, n)
    noise_candidates = {f"cand_noise_{i:04d}": rng.normal(0, 1, n) for i in range(n_noise_features)}

    y = 80 + 5 * base_temp - 3 * base_pressure + 6 * hidden_defect_1 - 4 * hidden_defect_2 + rng.normal(0, 0.3, n)

    base_df = pd.DataFrame(
        {
            ID_COL: sample_id,
            TARGET_COL: y,
            SPLIT_COL: split,
            "base_temp": base_temp,
            "base_pressure": base_pressure,
        }
    )
    candidate_df = pd.DataFrame(
        {
            ID_COL: sample_id,
            "hidden_defect_1": hidden_defect_1,
            "hidden_defect_2": hidden_defect_2,
            **noise_candidates,
        }
    )

    BASE_DATASET_PATH = demo_dir / "base_dataset.csv"
    CANDIDATE_FEATURES_PATH = demo_dir / "candidate_features.csv"
    BASE_FEATURE_COLS_PATH = demo_dir / "base_feature_cols.txt"
    base_df.to_csv(BASE_DATASET_PATH, index=False, encoding="utf-8-sig")
    candidate_df.to_csv(CANDIDATE_FEATURES_PATH, index=False, encoding="utf-8-sig")
    BASE_FEATURE_COLS_PATH.write_text("base_temp\nbase_pressure\n", encoding="utf-8")

    bad1 = base_df.loc[hidden_defect_1 >= np.quantile(hidden_defect_1, 0.75), [ID_COL]]
    good1 = base_df.loc[hidden_defect_1 < np.quantile(hidden_defect_1, 0.50), [ID_COL]]
    bad2 = base_df.loc[hidden_defect_2 <= np.quantile(hidden_defect_2, 0.25), [ID_COL]]
    good2 = base_df.loc[hidden_defect_2 > np.quantile(hidden_defect_2, 0.50), [ID_COL]]
    bad3 = base_df.sample(120, random_state=42)[[ID_COL]]
    good3 = base_df.drop(bad3.index).sample(180, random_state=43)[[ID_COL]]

    bad1.to_csv(group_dir / "defect_1_bad.csv", index=False, encoding="utf-8-sig")
    good1.to_csv(group_dir / "defect_1_good.csv", index=False, encoding="utf-8-sig")
    bad2.to_csv(group_dir / "defect_2_bad.csv", index=False, encoding="utf-8-sig")
    good2.to_csv(group_dir / "defect_2_good.csv", index=False, encoding="utf-8-sig")
    bad3.to_csv(group_dir / "defect_3_bad.csv", index=False, encoding="utf-8-sig")
    good3.to_csv(group_dir / "defect_3_good.csv", index=False, encoding="utf-8-sig")

    DEFECTS = [
        {"defect_id": "defect_1", "bad_group_path": group_dir / "defect_1_bad.csv", "good_group_path": group_dir / "defect_1_good.csv"},
        {"defect_id": "defect_2", "bad_group_path": group_dir / "defect_2_bad.csv", "good_group_path": group_dir / "defect_2_good.csv"},
        {"defect_id": "defect_3", "bad_group_path": group_dir / "defect_3_bad.csv", "good_group_path": group_dir / "defect_3_good.csv"},
    ]
    print("demo data written to", demo_dir)
    print("demo wafers:", len(base_df), "| demo candidate features:", candidate_df.shape[1] - 1)

## 3. 데이터 로드 및 검증

In [ ]:
base_df = load_base_dataset(BASE_DATASET_PATH)
candidate_df = load_candidate_features(CANDIDATE_FEATURES_PATH)
base_feature_cols = load_base_feature_cols(BASE_FEATURE_COLS_PATH)
candidate_cols = candidate_feature_cols(candidate_df, ID_COL)

validate_input_columns(
    base_df,
    candidate_df,
    base_feature_cols,
    id_col=ID_COL,
    target_col=TARGET_COL,
    split_col=SPLIT_COL,
)

all_df = align_base_and_candidates(base_df, candidate_df, ID_COL)
train_df, valid_df, test_df = split_frame(all_df, SPLIT_COL)

print("base_df:", base_df.shape)
print("candidate_df:", candidate_df.shape)
print("merged:", all_df.shape)
print("base features:", len(base_feature_cols))
print("candidate features:", len(candidate_cols))
print(all_df[SPLIT_COL].value_counts())

## 4. defect별 bad/good group 로드

In [ ]:
defect_groups = {}
for defect in DEFECTS:
    defect_id = defect["defect_id"]
    bad_ids = load_group_ids(defect["bad_group_path"], ID_COL)
    good_ids = load_group_ids(defect["good_group_path"], ID_COL)
    warnings = validate_defect_groups(
        defect_id,
        bad_ids,
        good_ids,
        all_df,
        id_col=ID_COL,
        split_col=SPLIT_COL,
        min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
    )
    defect_groups[defect_id] = {"bad": bad_ids, "good": good_ids, "warnings": set(warnings)}
    print(defect_id, "bad=", len(bad_ids), "good=", len(good_ids), "warnings=", warnings)

## 5. Baseline CatBoost 학습 및 residual 계산

In [ ]:
baseline_model = train_baseline_model(
    train_df,
    valid_df,
    base_feature_cols,
    TARGET_COL,
    BASELINE_MODEL_PARAMS,
)

all_df = add_baseline_predictions(
    baseline_model,
    all_df,
    feature_cols=base_feature_cols,
    target_col=TARGET_COL,
)
train_df, valid_df, test_df = split_frame(all_df, SPLIT_COL)

baseline_metrics = metrics_by_split(all_df, target_col=TARGET_COL, pred_col="baseline_pred", split_col=SPLIT_COL)
baseline_summary = baseline_residual_summary(
    all_df,
    target_col=TARGET_COL,
    pred_col="baseline_pred",
    residual_col="baseline_residual",
    split_col=SPLIT_COL,
    id_col=ID_COL,
    defects=defect_groups,
)

write_csv(baseline_metrics, OUT_DIR / "baseline_metrics.csv")
write_csv(baseline_summary, OUT_DIR / "baseline_residual_summary.csv")

display(baseline_metrics)
display(baseline_summary.head(12))

## 6. defect별 residual feature boosting

각 candidate feature는 `x_j -> current residual` 단일 feature 모델로만 평가됩니다. 선택 기준은 validation bad group의 residual 감소입니다.

In [ ]:
booster = ResidualFeatureBooster(
    ResidualFeatureBoosterConfig(
        residual_model_params=RESIDUAL_MODEL_PARAMS,
        n_rounds=BOOSTING_N_ROUNDS,
        select_per_round=SELECT_PER_ROUND,
        main_metric="bad_rmse_reduction",
        min_improvement=MIN_IMPROVEMENT,
        use_test_for_selection=False,
        min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
    )
)

quality_frames = []
selected_frames = []
curve_frames = []

for defect in DEFECTS:
    defect_id = defect["defect_id"]
    groups = defect_groups[defect_id]
    quality = profile_candidate_features(
        all_df,
        candidate_cols,
        id_col=ID_COL,
        split_col=SPLIT_COL,
        bad_ids=groups["bad"],
        good_ids=groups["good"],
        config=FEATURE_FILTER,
        protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
    )
    quality.insert(0, "defect_id", defect_id)
    quality_frames.append(quality)

    if any(str(item).startswith("low_valid_bad_samples") for item in groups.get("warnings", set())):
        print(defect_id, "skipped: low valid bad samples")
        continue

    print(f"[{defect_id}] scoring {len(candidate_cols)} candidates")
    result = booster.run_for_defect(
        train_df=train_df,
        valid_df=valid_df,
        test_df=test_df,
        candidate_cols=candidate_cols,
        target_col=TARGET_COL,
        id_col=ID_COL,
        baseline_pred_col="baseline_pred",
        defect_id=defect_id,
        bad_sample_ids=groups["bad"],
        good_sample_ids=groups["good"],
        quality_summary=quality,
        output_dir=OUT_DIR / "rankings",
    )

    if not result.selected_features.empty:
        selected_frames.append(result.selected_features)
        display(result.selected_features)
    else:
        print(defect_id, "selected no feature")

    if not result.residual_curve.empty:
        curve_frames.append(result.residual_curve)

quality_summary = pd.concat(quality_frames, ignore_index=True) if quality_frames else pd.DataFrame()
selected_features = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame()
residual_curve = pd.concat(curve_frames, ignore_index=True) if curve_frames else pd.DataFrame()

write_csv(quality_summary, OUT_DIR / "candidate_quality_summary.csv")
write_csv(selected_features, OUT_DIR / "selected_features.csv")
write_csv(residual_curve, OUT_DIR / "residual_reduction_curve.csv")
plot_residual_curve(residual_curve, OUT_DIR)

display(selected_features)
display(residual_curve)

## 7. Final model: Xb + selected Xnew 재학습

In [ ]:
selected_cols = []
if not selected_features.empty:
    for feature in selected_features["feature_name"].dropna().astype(str):
        if feature in candidate_cols and feature not in selected_cols:
            selected_cols.append(feature)

final_feature_cols = base_feature_cols + selected_cols
print("selected_cols:", selected_cols)
print("final feature count:", len(final_feature_cols))

final_model = train_final_model(
    train_df,
    valid_df,
    feature_cols=final_feature_cols,
    target_col=TARGET_COL,
    catboost_params=FINAL_MODEL_PARAMS,
)
all_df["final_pred"] = predict_final(final_model, all_df, final_feature_cols)

final_metrics = pd.concat(
    [
        evaluate_model_by_groups(
            all_df,
            model_name="baseline",
            pred_col="baseline_pred",
            target_col=TARGET_COL,
            split_col=SPLIT_COL,
            id_col=ID_COL,
            defects=defect_groups,
        ),
        evaluate_model_by_groups(
            all_df,
            model_name="final",
            pred_col="final_pred",
            target_col=TARGET_COL,
            split_col=SPLIT_COL,
            id_col=ID_COL,
            defects=defect_groups,
        ),
    ],
    ignore_index=True,
)
write_csv(final_metrics, OUT_DIR / "final_model_metrics.csv")
display(final_metrics)

## 8. Optional SHAP 검증

SHAP은 원인 확정이 아니라, 선택 feature가 final model에서 실제로 사용되는지 확인하는 보조 검증입니다.

In [ ]:
if SHAP_ENABLED:
    shap_summary = compute_shap_summary(
        final_model,
        all_df[all_df[SPLIT_COL].astype(str) == "test"],
        feature_cols=final_feature_cols,
        id_col=ID_COL,
        defects=defect_groups,
        selected_features=selected_features,
        max_samples=SHAP_MAX_SAMPLES,
    )
else:
    shap_summary = pd.DataFrame([{"status": "disabled"}])

write_csv(shap_summary, OUT_DIR / "shap_summary.csv")
display(shap_summary.head(30))

## 9. 산출물 확인

In [ ]:
print("saved to:", OUT_DIR)
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT_DIR))